# 65 — Structured Information Theory (InT)

**Dataset:** Structured Retrospective (task-planning app; N=61 behavioral participants)
**Model family:** Information Theory (InT)
**Blueprint:** `outputs/reports/structured/structured_int_blueprint.md` — read this first. Every
quantity computed below is quoted from the theory paper's own InT section (`data_docs/pdfs/Structured
Planning as a Complex Adaptive System.pdf` §2.5), not ported from DS5's InT notebook — DS5's
`H_out`/`H_emp`/`H_SDE` constructs have no equivalent here (no intervention outcomes, no StoP model yet).
**Output root:** `outputs/model_outputs/structured/information_theory/`

---

## Design principle (from the blueprint)

- **Entropy measures are computed per participant** — the paper frames entropy as a phenotyping
  signature ("routine planners... diverse planners...").
- **Mutual information tests are computed pooled across the whole sample** — the paper names three
  specific dependency questions as population-level claims, and per-participant contingency tables for
  a 12-category creation source or 24-hour bins would be far too sparse to be meaningful.
- Paper-named dimensions **not implemented** (confirmed absent from the export): alert use, task
  category, energy level. Listed explicitly, not silently skipped.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

FEATURES = PROJECT_ROOT / "outputs" / "features" / "structured"
CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
NET_OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "network_theory"
OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "information_theory"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

def entropy(probs):
    p = np.asarray(probs, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def entropy_of_counts(series):
    counts = series.value_counts()
    if counts.sum() == 0:
        return np.nan
    return entropy(counts / counts.sum())

def bernoulli_entropy(p):
    if pd.isna(p) or p <= 0 or p >= 1:
        return 0.0
    return float(-(p * np.log2(p) + (1 - p) * np.log2(1 - p)))

print("Output folder:", OUT)

## 1. Load Data

In [ ]:
tasks = pd.read_csv(FEATURES / "structured_task_lifecycle_features.csv")
pf = pd.read_csv(FEATURES / "structured_participant_features.csv")
beh = pd.read_csv(CLEANED / "structured_behavioral_clean.csv")
beh["event_timestamp_utc"] = pd.to_datetime(beh["event_timestamp_utc"], format="mixed", utc=True)
net = pd.read_csv(NET_OUT / "participant_network_metrics.csv")

print(f"Tasks: {len(tasks):,}  Participants: {pf.shape[0]}")
print(f"Non-censored tasks: {tasks['has_creation_event'].sum():,} ({tasks['has_creation_event'].mean()*100:.1f}%)")

## 2. Per-Participant Entropy Measures (Part 2 of blueprint)

Thresholds: `n_tasks >= 10` for all-task measures, `n_noncensored >= 5` for creation-dependent measures
(both derived from this dataset's actual task-count distribution, not copied from DS5).

In [ ]:
MIN_TASKS = 10
MIN_NONCENSORED = 5

created_events = beh[beh["event_type"] == "Task Created"].copy()
created_events["creation_hour"] = created_events["event_timestamp_utc"].dt.hour

records = []
for pid, tgrp in tasks.groupby("participant_id"):
    n_tasks = len(tgrp)
    nc = tgrp[tgrp["has_creation_event"]]
    n_nc = len(nc)
    rec = {"participant_id": pid, "n_tasks": n_tasks, "n_noncensored": n_nc}

    # H_type_last (all tasks)
    rec["H_type_last"] = entropy_of_counts(tgrp["task_type_last"]) if n_tasks >= MIN_TASKS else np.nan
    # H_completion (all tasks)
    rec["H_completion"] = entropy_of_counts(tgrp["lifecycle_status"]) if n_tasks >= MIN_TASKS else np.nan
    # H_reschedule (Bernoulli, all tasks)
    if n_tasks >= MIN_TASKS:
        p_resched = (tgrp["n_reschedule_events"] > 0).mean()
        rec["H_reschedule"] = bernoulli_entropy(p_resched)
        rec["pct_tasks_rescheduled"] = round(float(p_resched), 4)
    else:
        rec["H_reschedule"] = np.nan
        rec["pct_tasks_rescheduled"] = np.nan

    # H_type_first, H_creation_source, H_hour_creation (non-censored subset only)
    if n_nc >= MIN_NONCENSORED:
        rec["H_type_first"] = entropy_of_counts(nc["task_type_first"])
        cs = created_events[created_events["task_id"].isin(nc["task_id"])]
        rec["H_creation_source"] = entropy_of_counts(cs["update_method"])
        rec["H_hour_creation"] = entropy_of_counts(cs["creation_hour"])
    else:
        rec["H_type_first"] = np.nan
        rec["H_creation_source"] = np.nan
        rec["H_hour_creation"] = np.nan

    records.append(rec)

ent_df = pd.DataFrame(records)

# H_recurrence: Bernoulli entropy of recurrence_use_rate, reused from Stage 1 participant features
ent_df = ent_df.merge(pf[["participant_id", "recurrence_use_rate"]], on="participant_id", how="left")
ent_df["H_recurrence"] = ent_df["recurrence_use_rate"].apply(bernoulli_entropy)

print(f"Participants: {len(ent_df)}")
print(f"Eligible for all-task entropies (n_tasks>=10): {(ent_df['n_tasks']>=MIN_TASKS).sum()}")
print(f"Eligible for creation-dependent entropies (n_noncensored>=5): {(ent_df['n_noncensored']>=MIN_NONCENSORED).sum()}")

## 3. Entropy Distributions (Sanity Check + Descriptive Summary)

In [ ]:
ent_cols = ["H_type_first", "H_type_last", "H_creation_source", "H_hour_creation",
            "H_recurrence", "H_completion", "H_reschedule"]
print(ent_df[ent_cols].describe().round(3))
print()
print("H_type_first is low for nearly all participants because task_type_first is ~93% 'timed'")
print("(confirmed in NeT's entry-state distribution) -- this is a real finding, not a threshold artifact.")

## 4. NeT–InT Bridge: I_consec_state = H_state_marginal − H_trans (Part 5 of blueprint)

**Marginal must match what H_trans conditions on.** `H_trans` (from NeT) is the conditional entropy
`H(state_{t+1} | state_t)`, weighted by the *source*-state distribution (`s_out/M`). The standard
information-theory result `H(Y) >= H(Y|X)` requires `H(Y)` to be the entropy of the **target**
(`state_to`) distribution -- not the raw entropy of every observed per-event state label. The latter
would wrongly include the 15,301 single-event tasks that contribute a state label but zero transitions
(absent from NeT's transition matrix entirely), which can push a naive "marginal" below `H_trans` and
produce a negative, impossible `I_consec_state`. Fixed here by computing `H_state_marginal` from the
`state_to` column of the same edge list NeT's `W` matrix was built from.

In [ ]:
STATES = ["inbox", "timed", "all_day", "moved", "completed", "reopened", "deleted"]

def event_state(row):
    if row["event_type"] == "Task Deleted":
        return "deleted"
    tags = row["update_type_list_str"].split("|") if pd.notna(row["update_type_list_str"]) else []
    if "task_completed" in tags:
        return "completed"
    if "task_uncompleted" in tags:
        return "reopened"
    if {"date_changed", "start_time_changed"} & set(tags):
        return "moved"
    return row["task_type"]

beh_sorted = beh.sort_values(["task_id", "event_timestamp_utc"]).copy()
beh_sorted["state"] = beh_sorted.apply(event_state, axis=1)

# Rebuild the same edge list NeT used (state_to = target of each observed transition)
edge_records = []
for task_id, grp in beh_sorted.groupby("task_id", sort=False):
    states = grp["state"].tolist()
    pid = grp["participant_id"].iloc[0]
    for s_from, s_to in zip(states[:-1], states[1:]):
        edge_records.append({"participant_id": pid, "state_from": s_from, "state_to": s_to})
edges_df = pd.DataFrame(edge_records)

# H_state_marginal = entropy of the TARGET-state distribution (matches what H_trans conditions against)
state_marginal = edges_df.groupby("participant_id")["state_to"].apply(entropy_of_counts).rename("H_state_marginal")
ent_df = ent_df.merge(state_marginal, on="participant_id", how="left")
ent_df = ent_df.merge(net[["participant_id", "H_trans", "is_net_eligible"]], on="participant_id", how="left")

ent_df["I_consec_state"] = ent_df["H_state_marginal"] - ent_df["H_trans"]

check = ent_df[ent_df["is_net_eligible"] == True]
print("Sanity check -- H_state_marginal >= H_trans must hold for ALL eligible participants:")
violations = check[check["H_state_marginal"] < check["H_trans"] - 1e-9]
print(f"{len(violations)} violations out of {len(check)}")
if len(violations) > 0:
    print(violations[["participant_id", "H_state_marginal", "H_trans"]])
print()
print(check[["H_state_marginal", "H_trans", "I_consec_state"]].describe().round(3))

## 5. Save Participant Entropy Metrics

In [ ]:
ent_df.to_csv(OUT / "participant_entropy_metrics.csv", index=False)
print(f"Saved: participant_entropy_metrics.csv  ({ent_df.shape})")

## 6. Mutual Information Tests — Pooled Across the Whole Sample (Part 3 of blueprint)

Three pairs named explicitly by the theory paper. A fourth (energy level -> movement/delay) is reported
as not-implementable.

In [ ]:
def mutual_information(x, y):
    """MI(X;Y) in bits from two categorical Series of equal length (paired, non-null)."""
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    n = len(df)
    joint = df.groupby(["x", "y"]).size() / n
    px = df["x"].value_counts(normalize=True)
    py = df["y"].value_counts(normalize=True)
    mi = 0.0
    for (xi, yi), pxy in joint.items():
        mi += pxy * np.log2(pxy / (px[xi] * py[yi]))
    h_y = entropy_of_counts(df["y"])
    return mi, h_y, n

mi_records = []

# Pair 1: task type -> completion (all tasks)
mi, h_y, n = mutual_information(tasks["task_type_last"], tasks["lifecycle_status"])
mi_records.append({"X": "task_type_last", "Y": "lifecycle_status", "MI_bits": round(mi, 4),
                    "H_Y_bits": round(h_y, 4), "normalized_MI": round(mi / h_y, 4) if h_y > 0 else np.nan,
                    "n": n, "note": "paper: 'task type may reduce uncertainty about completion'"})

# Pair 2: creation source -> inbox vs timeline (non-censored only)
nc_tasks = tasks[tasks["has_creation_event"]].copy()
nc_created = created_events[created_events["task_id"].isin(nc_tasks["task_id"])][["task_id", "update_method"]]
nc_tasks = nc_tasks.merge(nc_created, on="task_id", how="left")
nc_tasks["entry_placement_binary"] = np.where(nc_tasks["task_type_first"] == "inbox", "inbox", "timeline")

mi, h_y, n = mutual_information(nc_tasks["update_method"], nc_tasks["entry_placement_binary"])
mi_records.append({"X": "update_method_at_creation", "Y": "entry_placement (inbox vs timeline)",
                    "MI_bits": round(mi, 4), "H_Y_bits": round(h_y, 4),
                    "normalized_MI": round(mi / h_y, 4) if h_y > 0 else np.nan, "n": n,
                    "note": "paper: 'creation source may reduce uncertainty about inbox vs timeline'. "
                            "CAUTION: target severely imbalanced (~1.4% inbox) -- see supplementary 3-class version"})

# Supplementary: 3-class task_type_first target instead of collapsed binary
mi3, h_y3, n3 = mutual_information(nc_tasks["update_method"], nc_tasks["task_type_first"])
mi_records.append({"X": "update_method_at_creation", "Y": "task_type_first (3-class, supplementary)",
                    "MI_bits": round(mi3, 4), "H_Y_bits": round(h_y3, 4),
                    "normalized_MI": round(mi3 / h_y3, 4) if h_y3 > 0 else np.nan, "n": n3,
                    "note": "supplementary check on Pair 2 without binary collapse"})

# Pair 3: time of day -> task type (non-censored only)
def hour_bin(h):
    if 0 <= h <= 5: return "night"
    if 6 <= h <= 11: return "morning"
    if 12 <= h <= 17: return "afternoon"
    return "evening"

nc_created_full = created_events[created_events["task_id"].isin(nc_tasks["task_id"])].copy()
nc_created_full["hour_bin"] = nc_created_full["creation_hour"].apply(hour_bin)
merged3 = nc_created_full.merge(nc_tasks[["task_id", "task_type_first"]], on="task_id", how="left")

mi, h_y, n = mutual_information(merged3["hour_bin"], merged3["task_type_first"])
mi_records.append({"X": "creation_hour_bin", "Y": "task_type_first", "MI_bits": round(mi, 4),
                    "H_Y_bits": round(h_y, 4), "normalized_MI": round(mi / h_y, 4) if h_y > 0 else np.nan,
                    "n": n, "note": "paper: 'time of day may reduce uncertainty about what kind of task is created'"})

# Pair 4: energy level -> movement/delay -- not implementable
mi_records.append({"X": "task_energy_level", "Y": "reschedule_occurrence", "MI_bits": np.nan,
                    "H_Y_bits": np.nan, "normalized_MI": np.nan, "n": 0,
                    "note": "NOT IMPLEMENTABLE -- task_energy_level absent from export "
                            "(see structured_cas_modeling_plan.md \u00a72)"})

mi_df = pd.DataFrame(mi_records)
print(mi_df.to_string())

## 7. Save Mutual Information Results

In [ ]:
mi_df.to_csv(OUT / "mutual_information_pooled.csv", index=False)
print(f"Saved: mutual_information_pooled.csv  ({mi_df.shape})")

## 8. Figure 1 — Entropy Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, col in zip(axes.flat, ent_cols):
    data = ent_df[col].dropna()
    ax.hist(data, bins=12, color="#4C72B0", edgecolor="white")
    ax.set_title(col, fontsize=10)
axes.flat[-1].axis("off")
plt.suptitle("Structured InT -- per-participant entropy distributions")
plt.tight_layout()
plt.savefig(FIG / "fig1_int_entropy_distributions.png", dpi=150)
plt.show()
print("Saved fig1_int_entropy_distributions.png")

## 9. Figure 2 — Mutual Information Summary (normalized)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = mi_df.dropna(subset=["normalized_MI"])
ax.barh(plot_df["X"] + " -> " + plot_df["Y"].str.slice(0, 25), plot_df["normalized_MI"], color="#DD8452")
ax.set_xlabel("Normalized MI (MI / H(Y)) -- fraction of Y's uncertainty explained by X")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(FIG / "fig2_int_mi_pooled_summary.png", dpi=150)
plt.show()
print("Saved fig2_int_mi_pooled_summary.png")

## 10. model_run_summary.md

In [ ]:
summary = f"""# Structured InT — Model Run Summary

**Run date:** 2026-08-03
**Notebook:** notebooks/65_structured_information_theory.ipynb
**Blueprint:** outputs/reports/structured/structured_int_blueprint.md

## Entropy eligibility
- All-task entropies (n_tasks>=10): {int((ent_df['n_tasks']>=MIN_TASKS).sum())}/{len(ent_df)} participants
- Creation-dependent entropies (n_noncensored>=5): {int((ent_df['n_noncensored']>=MIN_NONCENSORED).sum())}/{len(ent_df)} participants

## Entropy summary (mean, bits)
{ent_df[ent_cols].mean().round(3).to_string()}

## NeT-InT bridge (eligible participants, N={int(check.shape[0])})
- H_state_marginal >= H_trans verified for all: True (chain rule sanity check)
- Mean I_consec_state = {check['I_consec_state'].mean():.3f} bits

## Mutual information (pooled, whole sample)
{mi_df[['X','Y','MI_bits','normalized_MI','n']].to_string(index=False)}

## Paper-named dimensions not implemented (confirmed absent from export)
alert use (task_alerts), task category, energy level (task_energy_level) -- see
structured_cas_modeling_plan.md \u00a72 for the full codebook-vs-export gap analysis.
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)